# Moneli — Prédictions 2026 & Visualisations finales
Charge le modèle d'Arthur, prédit les prix 2026, affiche les courbes.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pickle, os, sys

sys.path.insert(0, os.path.dirname(os.getcwd()))
import config
print('✅ config.py chargé')

## 1. Charger les données réelles 2024 + 2025

In [ ]:
df24 = pd.read_csv(config.CSV_2024)
df25 = pd.read_csv(config.CSV_2025)
df   = pd.concat([df24, df25], ignore_index=True)

df = df[df[config.COL_TYPE].isin(config.TYPES_BIENS)].copy()
df = df.dropna(subset=[config.COL_PRIX, config.COL_SURFACE])
df['date_mutation'] = pd.to_datetime(df[config.COL_DATE])
df['annee']         = df['date_mutation'].dt.year
df['mois']          = df['date_mutation'].dt.month
df['prix_m2']       = df[config.COL_PRIX] / df[config.COL_SURFACE]
df = df[(df['prix_m2'] > 200) & (df['prix_m2'] < 15000)]

stats_reels = df.groupby([config.COL_TYPE, 'annee']).agg(
    prix_moyen = (config.COL_PRIX, 'mean'),
    prix_m2    = ('prix_m2',       'mean'),
).round(0).reset_index()

print(stats_reels.to_string(index=False))

## 2. Générer les scénarios 2026

In [ ]:
LAT_ANGERS = 47.478419
LON_ANGERS = -0.563166

np.random.seed(42)
types_config = {
    'Maison':      {'surfaces': [60, 97, 130, 180], 'pieces': [3, 4, 5, 6], 'terrain': [300, 500, 800, 1200]},
    'Appartement': {'surfaces': [30, 50, 70, 100],  'pieces': [1, 2, 3, 4], 'terrain': [0,   0,   0,   0]},
}

scenarios = []
for type_bien, cfg in types_config.items():
    for i in range(len(cfg['surfaces'])):
        lat = LAT_ANGERS + np.random.uniform(-0.5, 0.5)
        lon = LON_ANGERS + np.random.uniform(-0.5, 0.5)
        scenarios.append({
            config.COL_TYPE:    type_bien,
            config.COL_SURFACE: cfg['surfaces'][i],
            config.COL_PIECES:  cfg['pieces'][i],
            config.COL_TERRAIN: cfg['terrain'][i],
            config.COL_LAT:     round(lat, 6),
            config.COL_LON:     round(lon, 6),
            'type_encode':      1 if type_bien == 'Maison' else 0,
            'annee':            config.ANNEE_PRED,
            'mois':             6,
        })

X_2026 = pd.DataFrame(scenarios)
X_2026.to_csv(config.X_2026, index=False)
print(f'✅ {len(X_2026)} scénarios générés et exportés')

## 3. Charger le modèle d'Arthur et prédire

In [ ]:
with open(config.MODEL_OBJ1, 'rb') as f:
    model = pickle.load(f)

# Colonnes utilisées par Arthur
features_model = list(model.feature_names_in_)
print('Features du modèle Arthur :', features_model)

X_pred = X_2026[features_model].fillna(0)
X_2026['prix_predit'] = model.predict(X_pred)

X_2026.to_csv(config.PREDICTIONS, index=False)
print('\n✅ Prédictions exportées')
print(X_2026[[config.COL_TYPE, config.COL_SURFACE, 'prix_predit']].to_string(index=False))

## 4. Résumé des prédictions 2026

In [ ]:
resume_2026 = X_2026.groupby(config.COL_TYPE).agg(
    prix_predit_moyen = ('prix_predit', 'mean'),
).round(0)
resume_2026['prix_m2_predit'] = (resume_2026['prix_predit_moyen'] /
    X_2026.groupby(config.COL_TYPE)[config.COL_SURFACE].mean()).round(0)

print('=== Prédictions 2026 ===')
print(resume_2026.to_string())

pred_maison = resume_2026.loc['Maison', 'prix_predit_moyen']
pred_appart = resume_2026.loc['Appartement', 'prix_predit_moyen']

## 5. Courbe principale 2024 → 2025 → 2026

In [ ]:
COULEURS = {'Maison': '#0D9488', 'Appartement': '#6366F1'}
ANNEES   = [2024, 2025, 2026]

def get_prix(type_bien, annee):
    row = stats_reels[(stats_reels[config.COL_TYPE]==type_bien) & (stats_reels['annee']==annee)]
    return row['prix_moyen'].values[0] if len(row) else None

data_graph = {
    'Maison':      [get_prix('Maison', 2024),      get_prix('Maison', 2025),      pred_maison],
    'Appartement': [get_prix('Appartement', 2024), get_prix('Appartement', 2025), pred_appart],
}

fig, ax = plt.subplots(figsize=(11, 6))

for type_bien, valeurs in data_graph.items():
    c = COULEURS[type_bien]
    ax.plot([2024, 2025], valeurs[:2], color=c, linewidth=2.5, marker='o', markersize=9, zorder=3)
    ax.plot([2025, 2026], valeurs[1:], color=c, linewidth=2.5, linestyle='--', marker='o',
            markersize=9, markerfacecolor='white', markeredgewidth=2.5, zorder=3, label=type_bien)
    for x, y in zip(ANNEES, valeurs):
        suffix = '\n(ML)' if x == 2026 else ''
        ax.annotate(f'{y:,.0f} €{suffix}', (x, y),
                    textcoords='offset points', xytext=(0, 15),
                    ha='center', fontsize=10, color=c,
                    fontweight='bold' if x == 2026 else 'normal')

ax.axvspan(2025.5, 2026.4, alpha=0.07, color='gray')
ax.text(2026, ax.get_ylim()[0], 'prédiction', ha='center', fontsize=9,
        color='gray', style='italic', va='bottom')

ligne_plein   = plt.Line2D([0],[0], color='gray', linewidth=2,              label='Données réelles')
ligne_pointil = plt.Line2D([0],[0], color='gray', linewidth=2, linestyle='--', label='Prédiction ML')
patches = [mpatches.Patch(color=c, label=t) for t, c in COULEURS.items()]
ax.legend(handles=patches + [ligne_plein, ligne_pointil], loc='upper left', fontsize=10)

ax.set_title('Évolution du prix moyen — Maine-et-Loire (Dept. 49)', fontsize=14, fontweight='bold', pad=20)
ax.set_xlabel('Année', fontsize=12)
ax.set_ylabel('Prix moyen (€)', fontsize=12)
ax.set_xticks(ANNEES)
ax.grid(axis='y', alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig(os.path.join(config.BASE_DIR, 'moneli_courbe_finale.png'), dpi=150, bbox_inches='tight')
plt.show()
print('✅ Graphique sauvegardé : moneli_courbe_finale.png')

## 6. Prix au m² 2024 → 2025 → 2026

In [ ]:
def get_m2(type_bien, annee):
    row = stats_reels[(stats_reels[config.COL_TYPE]==type_bien) & (stats_reels['annee']==annee)]
    return row['prix_m2'].values[0] if len(row) else None

pred_m_m2 = resume_2026.loc['Maison',      'prix_m2_predit']
pred_a_m2 = resume_2026.loc['Appartement', 'prix_m2_predit']

data_m2 = {
    'Maison':      [get_m2('Maison', 2024),      get_m2('Maison', 2025),      pred_m_m2],
    'Appartement': [get_m2('Appartement', 2024), get_m2('Appartement', 2025), pred_a_m2],
}

fig, ax = plt.subplots(figsize=(11, 6))
for type_bien, valeurs in data_m2.items():
    c = COULEURS[type_bien]
    ax.plot([2024, 2025], valeurs[:2], color=c, linewidth=2.5, marker='o', markersize=9)
    ax.plot([2025, 2026], valeurs[1:], color=c, linewidth=2.5, linestyle='--', marker='o',
            markersize=9, markerfacecolor='white', markeredgewidth=2.5, label=type_bien)
    for x, y in zip(ANNEES, valeurs):
        suffix = '\n(ML)' if x == 2026 else ''
        ax.annotate(f'{y:,.0f} €/m²{suffix}', (x, y),
                    textcoords='offset points', xytext=(0, 15),
                    ha='center', fontsize=10, color=c)

ax.axvspan(2025.5, 2026.4, alpha=0.07, color='gray')
ax.legend(handles=patches + [ligne_plein, ligne_pointil], loc='upper left', fontsize=10)
ax.set_title('Évolution du prix au m² — Maine-et-Loire (Dept. 49)', fontsize=14, fontweight='bold', pad=20)
ax.set_xlabel('Année', fontsize=12)
ax.set_ylabel('Prix moyen au m² (€/m²)', fontsize=12)
ax.set_xticks(ANNEES)
ax.grid(axis='y', alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig(os.path.join(config.BASE_DIR, 'moneli_courbe_m2.png'), dpi=150, bbox_inches='tight')
plt.show()
print('✅ Graphique sauvegardé : moneli_courbe_m2.png')